In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
import time

# 1. Load Data
train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')
test_ids = test['PassengerId']
target = train['Transported']

# 2. Preprocessing & Feature Engineering
train['is_train'] = 1
test['is_train'] = 0
data = pd.concat([train.drop('Transported', axis=1), test])

amenities = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
data[amenities] = data[amenities].fillna(0)
data['TotalSpend'] = data[amenities].sum(axis=1)
data['NoSpend'] = (data['TotalSpend'] == 0).astype(int)

data['CryoSleep'] = data['CryoSleep'].map({True: 1, False: 0})
data.loc[(data['CryoSleep'].isna()) & (data['NoSpend'] == 1), 'CryoSleep'] = 1
data.loc[(data['CryoSleep'].isna()) & (data['NoSpend'] == 0), 'CryoSleep'] = 0

data[['Deck', 'CabinNum', 'Side']] = data['Cabin'].str.split('/', expand=True)
data['CabinNum'] = pd.to_numeric(data['CabinNum'], errors='coerce')

data['Group'] = data['PassengerId'].apply(lambda x: x.split('_')[0])
data['GroupSize'] = data.groupby('Group')['PassengerId'].transform('count')

data['Age'] = data['Age'].fillna(data['Age'].median())
data['CabinNum'] = data['CabinNum'].fillna(data['CabinNum'].median())
for col in ['HomePlanet', 'Destination', 'Deck', 'Side', 'VIP']:
    data[col] = data[col].fillna(data[col].mode()[0])

data['VIP'] = data['VIP'].map({True: 1, False: 0}).fillna(0).astype(int)
data = data.drop(['Name', 'Cabin', 'PassengerId', 'Group'], axis=1)

cat_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']
for col in cat_cols:
    data[col] = data[col].astype('category')

train_proc = data[data['is_train'] == 1].drop('is_train', axis=1)
test_proc = data[data['is_train'] == 0].drop('is_train', axis=1)

# 3. Cross-Validation & Metric Tracking
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_proc))
oof_probs = np.zeros(len(train_proc))
test_preds = np.zeros(len(test_proc))

params = {
    'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 1000,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'binary:logistic',
    'eval_metric': 'logloss', 'enable_categorical': True, 'tree_method': 'hist',
    'random_state': 42, 'early_stopping_rounds': 50
}

print("Training models and calculating metrics...")
fold_accuracies = []
start_time = time.time()

for train_idx, val_idx in skf.split(train_proc, target):
    X_train, X_val = train_proc.iloc[train_idx], train_proc.iloc[val_idx]
    y_train, y_val = target.iloc[train_idx], target.iloc[val_idx]
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    probs = model.predict_proba(X_val)[:, 1]
    preds = (probs > 0.5).astype(bool)
    
    oof_preds[val_idx] = preds
    oof_probs[val_idx] = probs
    test_preds += model.predict_proba(test_proc)[:, 1] / 5
    
    fold_accuracies.append(accuracy_score(y_val, preds))

training_time = time.time() - start_time

# 4. Calculate Final Metrics
print("\n" + "="*40)
print("FINAL METRICS FOR YOUR DOCUMENT")
print("="*40)

print(f"Model / algorithm: XGBoost (Extreme Gradient Boosting)")
print(f"Libraries / frameworks: pandas, numpy, scikit-learn, xgboost")
print(f"Final feature count: {train_proc.shape[1]}")
print(f"CV strategy: 5-Fold Stratified Cross-Validation")
print(f"CV Accuracy (mean): {np.mean(fold_accuracies):.4f}")
print(f"CV Accuracy (std): {np.std(fold_accuracies):.4f}")
print(f"Validation Accuracy (Overall OOF): {accuracy_score(target, oof_preds):.4f}")
print(f"Precision: {precision_score(target, oof_preds):.4f}")
print(f"Recall: {recall_score(target, oof_preds):.4f}")
print(f"F1 score: {f1_score(target, oof_preds):.4f}")
print(f"ROC-AUC: {roc_auc_score(target, oof_probs):.4f}")
print(f"Log loss: {log_loss(target, oof_probs):.4f}")
print(f"Random seed set? (Y/N): Y (random_state=42)")
print(f"Training time: {training_time:.2f} seconds")

# 5. Save Submission
final_predictions = (test_preds > 0.5).astype(bool)
submission = pd.DataFrame({'PassengerId': test_ids, 'Transported': final_predictions})
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved to 'submission.csv'!")

/tmp/ipykernel_59/3767888478.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].fillna(data[col].mode()[0])


Training models and calculating metrics...

FINAL METRICS FOR YOUR DOCUMENT
Model / algorithm: XGBoost (Extreme Gradient Boosting)
Libraries / frameworks: pandas, numpy, scikit-learn, xgboost
Final feature count: 16
CV strategy: 5-Fold Stratified Cross-Validation
CV Accuracy (mean): 0.8113
CV Accuracy (std): 0.0062
Validation Accuracy (Overall OOF): 0.8113
Precision: 0.8124
Recall: 0.8132
F1 score: 0.8128
ROC-AUC: 0.9059
Log loss: 0.3789
Random seed set? (Y/N): Y (random_state=42)
Training time: 3.39 seconds

Submission saved to 'submission.csv'!


In [3]:
importance = catb_model.get_feature_importance()
feature_names = X_train.columns

df_importance = (
                pd.DataFrame({'Feature': feature_names, 'Importance': importance}).sort_values(by='Importance', ascending=False)
)

plt.figure(figsize = (10,6))
sns.barplot(data=df_importance, x='Importance', y='Feature', hue='Feature', palette = 'viridis')
plt.title("Top 10 Features")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.show()

NameError: name 'catb_model' is not defined